# EDA: gravity and spacing (analytics/gravity)

Tables from `analytics.gravity.eda` over the 1 Hz player-frames inside possessions. With `DATA_ROOT = None` the games come from the fixture generator (`tests/_fx_gravity.py`, or `analytics.fixtures` when present) and **every table is fixture output, not a result**. Set `DATA_ROOT` to the owner's data root to run on real games (no number from a held-out game is used to fit anything here; the EDA reads training games only).

Output is tables and markdown only (no plotting library in the environment).

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name and not (REPO / "analytics" / "gravity").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from analytics.gravity import eda, features as F, validate as V

DATA_ROOT = None  # e.g. "/Volumes/data/nba" on the Mac
SEED, N_GAMES = 0, 6
if DATA_ROOT:
    games = [d for d in V.load_data_root(DATA_ROOT) if not F.is_heldout(d)]
    label = "real training games"
else:
    from tests._fx_gravity import SOURCE, synth_season
    games = [d for d in synth_season(SEED, n_games=N_GAMES, heldout_every=0)]
    label = "fixture output, not a result (%s)" % SOURCE
games, counts = V.validate_games(games)
print(label, ":", len(games), "games")

## Sampling counts per game

In [ ]:
tables = eda.run(games, seed=SEED)
print(label)
print(eda.md_table(tables["counts"]))

## Nearest-defender distance by distance to the basket and by has_ball

The target by the two features that should carry most of it: defenders sag off players far from the basket and close on the handler.

In [ ]:
print(label)
print(eda.md_table(tables["nearest_def_by_distance_bin"]))
print()
print(eda.md_table(tables["nearest_def_by_has_ball"]))

## Hull area by possession end_kind

Spacing (convex hull of the five offensive players, square feet) and the mean nearest-defender distance, joined to the possession's end_kind for exploration only (end_kind is a target column and never enters a feature frame).

In [ ]:
print(label)
print(eda.md_table(tables["hull_area_by_end_kind"], nd=1))

## Correlation table

Pearson correlation of each feature with the target and with the distance to the basket (collinearity check for the ridge model).

In [ ]:
print(label)
print(eda.md_table(tables["correlation_table"]))

## Shuffled-y check

Ridge out of fold by game on the real target and on a permuted target. The permuted run should sit at R2 about 0; a feature that leaks the target would score well on both.

In [ ]:
print(label)
print(eda.md_table(tables["shuffled_y_check"], nd=4))